# S6E10 — 20 Own TabPFN-3.5 with our features, largest context that fits (Kaggle GPU)

> **Pipeline position:** 08 public TabPFN members, 19 own TabPFN probe (200k context) → **20 own TabPFN, all folds, on Kaggle** → 11 stack

| | |
|---|---|
| **Where** | Kaggle notebook, **GPU T4 ×2** (works with one GPU too, slower), **Internet on** |
| **Inputs (Add Input)** | competition `playground-series-s6e10`; dataset `deonissx/s6e10-orig-data` (`orig_data.csv`); optionally the output of a previous version of this notebook (to resume) |
| **Secret** | `TABPFN_TOKEN` (Add-ons → Secrets): the Prior Labs API key; the licence must already be accepted on ux.priorlabs.ai |
| **Output** | `/kaggle/working/tabpfn_own_full/oof_<k>.npy`, `test_<k>.npy`, `meta_<k>.json` per fold; when all 5 exist: `oof_tabpfn_own_full.npy`, `test_tabpfn_own_full.npy` (probabilities, train / test row order of the competition files) |
| **Runtime** | estimate ≈ 1 h per fold on a T4 at full context; the two GPUs take folds in parallel → ≈ 3–4 h. Run as *Save Version → Save & Run All (Commit)* |

**Why:** in `19` our own TabPFN with our features beat the same TabPFN without them by +0.0003 (fold 0, 200k context), but at 200k it is 0.001 weaker than the public member that used the full 560k context, so the stack ignored it. Here the context is as large as the GPU allows (the full fold if it fits), on all 5 folds, so the member can be judged by the real nested CV of `11`.

**Features (variant `ours` of `19`):** 21 raw columns (categoricals as codes, missing arrival delay = −1, `Flight Distance` declared categorical) + in-fold target encoding of the route and of route × class + the two captains trained on the original survey (`orig_logit` as in `03`, `og_cat_logit` as in `09`, recomputed here and checked against their reference AUCs).

**Leakage guards:** the folds are the project's `StratifiedKFold(5, shuffle=True, random_state=42)` on `train.csv` order. The target encoder is fitted per fold on the training part only (inner 5-fold cross-fitting for the context rows) and only applied to validation and test. The captains never see competition labels. The context is drawn from the fold's training part only.

**Robustness:** each finished fold is saved immediately. If the session dies, add this notebook's previous output as an input and run again: finished folds are copied back and skipped. On a CUDA out-of-memory error the context is reduced step by step (and the prediction batch halved) instead of failing.

> **Glossary:** *Context* = the labelled rows TabPFN reads before predicting. *KV cache* = the stored representation of the context (the GPU-memory bottleneck). *Captain* = model trained on the original survey whose prediction is a feature.

In [ ]:
import subprocess, sys, os, glob, json, time, shutil
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tabpfn==9.1.0'], check=False)
import torch          # first torch import before numpy/sklearn (project convention)
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
import warnings; warnings.filterwarnings('ignore')

# Licence key from Kaggle Secrets -> environment variable read by tabpfn (inherited by the GPU worker processes).
ON_KAGGLE = Path('/kaggle/input').exists()
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['TABPFN_TOKEN'] = UserSecretsClient().get_secret('TABPFN_TOKEN')
    print('TABPFN_TOKEN loaded from Kaggle Secrets')
except Exception as e:
    print('secret TABPFN_TOKEN not available:', str(e)[:160])
    if ON_KAGGLE:
        raise SystemExit('STOP: add the secret TABPFN_TOKEN (Add-ons -> Secrets) and tick "Attached" for this notebook, then run again.')

CTX_TRY = [560_000, 420_000, 320_000, 240_000, 200_000]   # context sizes tried in this order (560k >= a full fold: all training rows)
MAX_HOURS = 11.0                                           # a worker does not start a new fold after this (Kaggle limit: 12 h)
N_GPU = torch.cuda.device_count()
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
OUT = WORK / 'tabpfn_own_full'; OUT.mkdir(exist_ok=True); (WORK / 'logs').mkdir(exist_ok=True)
def find(name, roots=('/kaggle/input', '../data/raw')):
    # os.walk with followlinks: Kaggle mounts inputs under /kaggle/input/competitions/... and /kaggle/input/datasets/... (sometimes as links)
    for r in roots:
        for dirpath, _, files in os.walk(r, followlinks=True):
            if name in files: return Path(dirpath) / name
    return None
tr_path, orig_path = find('train.csv'), find('orig_data.csv')
if tr_path is None or orig_path is None:
    print('found under /kaggle/input:', [str(p) for p in Path('/kaggle/input').rglob('*.csv')][:20] if ON_KAGGLE else '-')
    raise SystemExit(f'STOP: missing input -> train.csv: {tr_path} | orig_data.csv: {orig_path}. Add Input: competition playground-series-s6e10 and the dataset with orig_data.csv.')
comp = tr_path.parent
print('GPUs:', N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)], '| competition:', comp, '| original:', orig_path)

## 1. Data and the two captains (recomputed; reference AUCs 0.95472 and 0.9544)
Same code as `03` (LightGBM captain) and `09` (CatBoost captain), trained on the original survey only.

In [ ]:
import lightgbm as lgb
from catboost import CatBoostClassifier
train = pd.read_csv(comp / 'train.csv'); test = pd.read_csv(comp / 'test.csv'); orig = pd.read_csv(orig_path)
tobin = lambda s: s.astype(str).str.strip().str.lower().isin(['true', '1', '1.0', 'satisfied']).astype(int)
y = tobin(train['satisfaction']).values
CATS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
FEATS = [c for c in test.columns if c != 'id']
assert len(FEATS) == 21 and len(train) == 699635 and len(test) == 299844

# --- captain 1 (03): regularised LightGBM, arrival delay filled with departure delay
def low(d): d = d.copy(); d.columns = d.columns.str.lower().str.replace(' ', '_'); return d
tr_l, te_l, og_l = low(train), low(test), low(orig)
for d in (tr_l, te_l, og_l): d['arrival_delay_in_minutes'] = d['arrival_delay_in_minutes'].fillna(d['departure_delay_in_minutes'])
feats_l = [c for c in tr_l.columns if c in og_l.columns and c not in ('id', 'satisfaction')]
cats_l = ['gender', 'customer_type', 'type_of_travel', 'class']
A, Bt, Ct = og_l[feats_l].copy(), tr_l[feats_l].copy(), te_l[feats_l].copy()
for c in cats_l:
    cats = sorted(set(A[c].astype(str)) | set(Bt[c].astype(str)) | set(Ct[c].astype(str)))
    for d in (A, Bt, Ct): d[c] = pd.Categorical(d[c].astype(str), categories=cats)
m1 = lgb.LGBMClassifier(objective='binary', n_estimators=600, learning_rate=0.05, max_depth=8, min_child_samples=20, subsample=0.8,
                        colsample_bytree=0.5, reg_lambda=5.0, reg_alpha=0.05, random_state=123, n_jobs=-1, verbosity=-1)
m1.fit(A, tobin(og_l['satisfaction']), categorical_feature=cats_l)
lgt = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / np.clip(1 - p, 1e-6, 1 - 1e-6))
orig_logit = np.r_[lgt(m1.predict_proba(Bt)[:, 1]), lgt(m1.predict_proba(Ct)[:, 1])].astype(np.float32)

# --- captain 2 (09): CatBoost on complete, de-duplicated original rows
BASE = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes'] + [c for c in FEATS if c not in CATS + ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']] + CATS
CAT_MAPS = {c: {v: i for i, v in enumerate(sorted(pd.concat([train[c], test[c]]).dropna().unique()))} for c in CATS}
def enc(df):
    X = df[BASE].copy()
    for c in CATS: X[c] = X[c].map(CAT_MAPS[c]).fillna(-1).astype('int16')
    return X
og = orig.dropna(subset=[c for c in BASE if c != 'Arrival Delay in Minutes'])[BASE + ['satisfaction']].drop_duplicates()
m2 = CatBoostClassifier(iterations=1500, learning_rate=0.05, depth=8, cat_features=CATS, random_seed=42, verbose=0, allow_writing_files=False, thread_count=-1)
m2.fit(enc(og), tobin(og['satisfaction']).values)
p2 = m2.predict_proba(enc(pd.concat([train[BASE], test[BASE]], ignore_index=True)))[:, 1].clip(1e-5, 1 - 1e-5)
og_cat_logit = np.log(p2 / (1 - p2)).astype(np.float32)
a1, a2 = roc_auc_score(y, orig_logit[:len(train)]), roc_auc_score(y, og_cat_logit[:len(train)])
print(f'captain AUCs on competition train: LightGBM {a1:.5f} (ref 0.95472) | CatBoost {a2:.5f} (ref 0.9544)')
assert abs(a1 - 0.95472) < 0.002 and abs(a2 - 0.9544) < 0.002, 'captain differs from the reference'

## 2. Feature matrix and per-fold target encoding
Everything a GPU worker needs is written to one `.npz` file: the fold-independent columns, and for every fold the target-encoded columns (training rows cross-fitted, validation and test rows transformed).

In [ ]:
full = pd.concat([train[FEATS], test[FEATS]], ignore_index=True)
for c in CATS: full[c] = full[c].astype('category').cat.codes
full['Arrival Delay in Minutes'] = full['Arrival Delay in Minutes'].fillna(-1)
BASEX = np.column_stack([full.values.astype(np.float32), orig_logit, og_cat_logit])
cat_idx = [full.columns.get_loc(c) for c in CATS] + [full.columns.get_loc('Flight Distance')]
n_tr = len(train)
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(n_tr), y))
fd = full['Flight Distance'].values.astype(int); cls = full['Class'].values.astype(int)
keys = pd.DataFrame({'fd': fd, 'fd_cls': fd * 10 + cls})
TE_TR, TE_TE = [], []
for tr, va in FOLDS:
    t_tr = np.full((n_tr, 2), np.nan, dtype=np.float32)
    enc_ = TargetEncoder(target_type='binary', cv=5, shuffle=True, random_state=42)
    t_tr[tr] = enc_.fit_transform(keys.iloc[tr], y[tr])       # cross-fitted on the fold's training part
    t_tr[va] = enc_.transform(keys.iloc[va])                  # validation: transform only
    TE_TR.append(t_tr); TE_TE.append(enc_.transform(keys.iloc[n_tr:]).astype(np.float32))
np.savez(WORK / 'tabpfn_inputs.npz', base=BASEX, y=y, te_tr=np.stack(TE_TR), te_te=np.stack(TE_TE),
         cat_idx=np.array(cat_idx), folds_va=np.array([va for _, va in FOLDS], dtype=object), n_tr=n_tr)
print('features per row:', BASEX.shape[1] + 2, '| categorical indices:', cat_idx)

## 3. Download the weights once, restore finished folds from a previous run

In [ ]:
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
_m = TabPFNClassifier.create_default_for_version(ModelVersion.V3_5, device='cpu'); _m.fit(np.random.rand(50, 3), np.arange(50) % 2); del _m
print('TabPFN-3.5 weights available')
for f in [str(p) for p in Path('/kaggle/input').rglob('tabpfn_own_full/*_*.*')] if ON_KAGGLE else []:
    if not (OUT / Path(f).name).exists(): shutil.copy(f, OUT); print('restored', Path(f).name)
print('finished folds:', sorted(int(p.stem.split('_')[1]) for p in OUT.glob('oof_*.npy')))

## 4. GPU workers
One process per GPU (`CUDA_VISIBLE_DEVICES`), each takes every second fold. A worker tries the context sizes of `CTX_TRY` in order and keeps the first that fits for its next folds.

In [ ]:
WORKER = r"""
import os, sys, json, time, gc
import torch, numpy as np
from sklearn.metrics import roc_auc_score
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
folds = [int(k) for k in sys.argv[1].split(',')]; ctx_try = [int(c) for c in sys.argv[2].split(',')]; out = sys.argv[3]; max_h = float(sys.argv[4])
D = np.load(sys.argv[5], allow_pickle=True); base, y, n_tr = D['base'], D['y'], int(D['n_tr']); cat_idx = D['cat_idx'].tolist()
t_start = time.time()
def log(*a): print(time.strftime('%H:%M:%S'), *a, flush=True)
def model():
    return TabPFNClassifier.create_default_for_version(ModelVersion.V3_5, device='cuda', n_estimators=1, random_state=0,
        ignore_pretraining_limits=True, fit_mode='fit_with_cache', kv_cache_precision='int8', inference_precision='autocast',
        memory_saving_mode='auto', categorical_features_indices=cat_idx)
def predict(m, Xq, batch):
    out_ = np.empty(len(Xq), dtype=np.float32); i = 0
    while i < len(Xq):
        try:
            out_[i:i + batch] = m.predict_proba(Xq[i:i + batch])[:, 1]; i += batch
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            if batch <= 256: raise
            batch //= 2; log('  prediction OOM -> batch', batch)
    return out_
for k in folds:
    if os.path.exists(f'{out}/oof_{k}.npy') and os.path.exists(f'{out}/test_{k}.npy'): log(f'fold {k}: already done'); continue
    if (time.time() - t_start) / 3600 > max_h: log(f'fold {k}: time limit reached, stop'); break
    va = D['folds_va'][k].astype(int); tr = np.setdiff1d(np.arange(n_tr), va)
    Xtr = np.column_stack([base[:n_tr], D['te_tr'][k]]); Xte = np.column_stack([base[n_tr:], D['te_te'][k]])
    for ctx in list(ctx_try):
        idx = tr if ctx >= len(tr) else np.random.RandomState(0).choice(tr, ctx, replace=False)   # context = fold TRAINING rows only
        m = None
        try:
            t0 = time.time(); torch.cuda.reset_peak_memory_stats()
            m = model(); m.fit(Xtr[idx], y[idx]); t_fit = time.time() - t0
            log(f'fold {k}: context {len(idx):,} fitted in {t_fit/60:.1f} min, peak {torch.cuda.max_memory_allocated()/2**30:.1f} GB')
            p_va = predict(m, Xtr[va], 2048); auc = roc_auc_score(y[va], p_va)
            log(f'fold {k}: validation AUC {auc:.5f} ({(time.time()-t0)/60:.1f} min so far)')
            p_te = predict(m, Xte, 2048)
            np.save(f'{out}/oof_{k}.npy', p_va); np.save(f'{out}/test_{k}.npy', p_te)
            json.dump({'fold': k, 'context': int(len(idx)), 'auc': float(auc), 'minutes': (time.time()-t0)/60}, open(f'{out}/meta_{k}.json', 'w'))
            log(f'fold {k}: saved ({(time.time()-t0)/60:.1f} min)')
            ctx_try = [c for c in ctx_try if c <= ctx]          # keep the size that worked for the next folds
            break
        except torch.OutOfMemoryError:
            log(f'fold {k}: context {len(idx):,} out of memory -> smaller')
        finally:
            m = None; gc.collect(); torch.cuda.empty_cache()
    else:
        log(f'fold {k}: no context size fitted')
log('worker done')
"""
(WORK / 'tabpfn_worker.py').write_text(WORKER)
todo = [k for k in range(5) if not (OUT / f'oof_{k}.npy').exists()]
groups = [todo[g::max(N_GPU, 1)] for g in range(max(N_GPU, 1))]
procs = []
for g, fl in enumerate(groups):
    if not fl: continue
    env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(g)}
    logf = open(WORK / 'logs' / f'gpu{g}.log', 'w')
    procs.append((g, subprocess.Popen([sys.executable, '-u', str(WORK / 'tabpfn_worker.py'), ','.join(map(str, fl)), ','.join(map(str, CTX_TRY)),
                                       str(OUT), str(MAX_HOURS), str(WORK / 'tabpfn_inputs.npz')], env=env, stdout=logf, stderr=subprocess.STDOUT)))
    print(f'GPU {g}: folds {fl}')
seen = {g: 0 for g, _ in procs}
while procs:
    time.sleep(60)
    for g, p in list(procs):
        lines = (WORK / 'logs' / f'gpu{g}.log').read_text(errors='ignore').splitlines()
        for ln in lines[seen[g]:]:
            if any(s in ln for s in ('fold', 'worker', 'Error', 'Traceback', 'OOM')): print(f'[GPU {g}]', ln, flush=True)
        seen[g] = len(lines)
        if p.poll() is not None: procs.remove((g, p)); print(f'[GPU {g}] exited with code {p.returncode}', flush=True)

## 5. Assemble (when all 5 folds exist)

In [ ]:
metas = [json.load(open(OUT / f'meta_{k}.json')) for k in range(5) if (OUT / f'meta_{k}.json').exists()]
print(pd.DataFrame(metas).round(5).to_string(index=False) if metas else 'no finished folds')
if all((OUT / f'oof_{k}.npy').exists() for k in range(5)):
    oof = np.full(n_tr, np.nan)
    for k, (_, va) in enumerate(FOLDS): oof[va] = np.load(OUT / f'oof_{k}.npy')
    tst = np.mean([np.load(OUT / f'test_{k}.npy') for k in range(5)], axis=0)
    assert not np.isnan(oof).any()
    np.save(WORK / 'oof_tabpfn_own_full.npy', oof); np.save(WORK / 'test_tabpfn_own_full.npy', tst)
    print(f'OOF AUC {roc_auc_score(y, oof):.5f} | per fold', [round(roc_auc_score(y[va], oof[va]), 5) for _, va in FOLDS])
    print('saved oof_tabpfn_own_full.npy / test_tabpfn_own_full.npy -> download them into preds/')
else:
    print('not all folds finished: add this version\'s output as an input and run again to resume')

## Results & decisions
*Filled in locally after the run (nested CV of the stack with this member, `11` procedure).*

## Glossary
* **Context** — labelled rows TabPFN reads before predicting; here as many of the fold's training rows as fit into GPU memory.
* **KV cache (int8)** — the stored representation of the context; 8-bit storage halves its memory.
* **Worker** — a separate Python process bound to one GPU.
* **Resume** — finished folds from a previous run's output are copied back and skipped.